# Redes neuronales recurrentes

En el módulo anterior usamos representaciones semánticas ricas del texto y un clasificador lineal simple sobre los embeddings. Esta arquitectura captura el significado agregado de las palabras de una oración, pero no toma en cuenta el **orden** de las palabras, porque la operación de agregación sobre los embeddings elimina esa información del texto original. Como estos modelos no pueden modelar el orden de las palabras, no son capaces de resolver tareas más complejas o ambiguas, como la generación de texto o la respuesta a preguntas.

Para capturar el significado de una secuencia de texto necesitamos otra arquitectura de red neuronal, llamada **red neuronal recurrente** o RNN. En una RNN, pasamos la oración por la red un símbolo a la vez, y la red produce un **estado**, que luego volvemos a entregar a la red junto con el siguiente símbolo.

<img alt="RNN" src="images/rnn.png" width="60%"/>

Dada una secuencia de tokens de entrada $X_0,\dots,X_n$, la RNN crea una secuencia de bloques de red neuronal y entrena esta secuencia de extremo a extremo mediante retropropagación. Cada bloque de la red recibe un par $(X_i,S_i)$ como entrada y produce $S_{i+1}$ como resultado. El estado final $S_n$ o la salida $X_n$ se entrega a un clasificador lineal para producir el resultado. Todos los bloques de la red comparten los mismos pesos y se entrenan de extremo a extremo con una sola pasada de retropropagación.

Como los vectores de estado $S_0,\dots,S_n$ se propagan a través de la red, esta es capaz de aprender las dependencias secuenciales entre palabras. Por ejemplo, cuando la palabra *not* (no) aparece en algún punto de la secuencia, la red puede aprender a negar ciertos elementos del vector de estado, produciendo así una negación.

> Como los pesos de todos los bloques RNN de la figura son compartidos, la misma figura puede representarse como un único bloque (a la derecha) con un lazo de retroalimentación recurrente, que devuelve el estado de salida de la red a la entrada.

Veamos cómo las redes neuronales recurrentes pueden ayudarnos a identificar el idioma de una oración.

In [1]:
import os
import random
import hashlib
import zipfile
import collections
import urllib.request
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando {device}")

Usando cuda


## El dataset

Usaremos el dataset de traducción automática inglés–francés del capítulo [*Machine Translation and the Dataset*](https://d2l.ai/chapter_recurrent-modern/machine-translation-and-dataset.html) del libro *Dive into Deep Learning* (d2l). Contiene unos 167 mil pares de oraciones bilingües del [proyecto Tatoeba](https://www.manythings.org/anki/): cada línea del archivo `fra.txt` tiene una oración en inglés, un tabulador y su traducción al francés.

Descargamos el archivo y lo preprocesamos igual que en d2l: reemplazamos los espacios no separables por espacios normales, pasamos todo a minúsculas e insertamos un espacio entre las palabras y los signos de puntuación.

In [2]:
DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip'
DATA_SHA1 = '94646ad1522d915e7b0f9296181140edcf86a4f5'

def download_extract(url=DATA_URL, sha1=DATA_SHA1, root='./data'):
    os.makedirs(root, exist_ok=True)
    fname = os.path.join(root, url.split('/')[-1])
    if not os.path.exists(fname):
        print(f"Descargando {url}...")
        urllib.request.urlretrieve(url, fname)
    with open(fname, 'rb') as f:
        assert hashlib.sha1(f.read()).hexdigest() == sha1, f"{fname} está corrupto, bórralo y vuelve a ejecutar"
    with zipfile.ZipFile(fname) as z:
        z.extractall(root)
    return os.path.join(root, 'fra-eng')

def preprocess(text):
    # reemplazar espacios no separables por espacios y pasar a minúsculas
    text = text.replace('\u202f', ' ').replace('\xa0', ' ').lower()
    # insertar un espacio entre las palabras y los signos de puntuación
    no_space = lambda char, prev_char: char in ',.!?' and prev_char != ' '
    return ''.join(' ' + c if i > 0 and no_space(c, text[i-1]) else c
                   for i, c in enumerate(text))

data_dir = download_extract()
with open(os.path.join(data_dir, 'fra.txt'), encoding='utf-8') as f:
    raw_text = f.read()
pairs = [line.split('\t') for line in preprocess(raw_text).split('\n')]
pairs = [p for p in pairs if len(p) == 2]
print(f"{len(pairs)} pares de oraciones")
pairs[:5]

Descargando http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip...


167130 pares de oraciones


[['go .', 'va !'],
 ['hi .', 'salut !'],
 ['run !', 'cours !'],
 ['run !', 'courez !'],
 ['who ?', 'qui ?']]

## La tarea: identificación de idioma

En la próxima unidad usaremos este dataset para lo que fue pensado: traducir de inglés a francés con un modelo *encoder–decoder*. Por ahora lo usaremos para una tarea de clasificación de secuencias más sencilla: **dada una oración, decidir si está escrita en inglés o en francés**. Como cada oración en inglés aparece junto a su traducción, ambas clases tienen exactamente el mismo contenido, de modo que el modelo tiene que aprender a reconocer el idioma y no el tema.

Separamos el 10% de los **pares** para prueba, para que la traducción de una oración de prueba nunca aparezca en el entrenamiento. Al igual que en d2l, el texto se tokeniza separando por espacios, y los tokens que aparecen menos de 2 veces se reemplazan por el token desconocido `<unk>`. El token `<pad>` (índice 0) se usa para rellenar las secuencias.

In [3]:
classes = ['inglés', 'francés']

random.seed(42)
random.shuffle(pairs)
n_test = len(pairs) // 10

def to_samples(ps):
    # pares (etiqueta, texto): 0 = inglés, 1 = francés
    return [(0, en) for en, fr in ps] + [(1, fr) for en, fr in ps]

train_dataset = to_samples(pairs[n_test:])
test_dataset = to_samples(pairs[:n_test])

def tokenizer(x):
    return [t for t in x.split(' ') if t]

counter = collections.Counter(t for _, x in train_dataset for t in tokenizer(x))
vocab = ['<pad>', '<unk>'] + [t for t, c in counter.most_common() if c >= 2]
stoi = {t: i for i, t in enumerate(vocab)}  # diccionario para convertir tokens en índices
vocab_size = len(vocab)

def encode(x):
    return [stoi.get(t, stoi['<unk>']) for t in tokenizer(x)]

print(f"Tamaño del vocabulario: {vocab_size}")
print(f"Entrenamiento: {len(train_dataset)} oraciones, prueba: {len(test_dataset)} oraciones")
for label, x in train_dataset[:2] + train_dataset[-2:]:
    print(f"**{classes[label]}** -> {x} -> {encode(x)}")

Tamaño del vocabulario: 25904
Entrenamiento: 300834 oraciones, prueba: 33426 oraciones
**inglés** -> why do you want such an old car ? -> [118, 29, 5, 54, 621, 146, 338, 221, 3]
**inglés** -> i can't always understand everything you say . -> [4, 95, 281, 391, 345, 5, 227, 2]
**francés** -> nous sommes tous tombés d'accord . -> [32, 192, 171, 8005, 750, 2]
**francés** -> y a-t-il qui que ce soit , là-dedans ? -> [101, 406, 69, 13, 27, 236, 14, 2027, 3]


Por último, definimos las funciones auxiliares que usaremos durante el notebook: `padify` arma un minibatch rellenando con ceros todas las secuencias hasta el largo de la más larga, y `train_epoch` / `train_epoch_emb` implementan el ciclo de entrenamiento estándar de PyTorch durante una época (como en el laboratorio 3.1), reportando la exactitud acumulada cada `report_freq` minibatches.

In [4]:
def padify(b):
    # b es una lista de tuplas (etiqueta, texto) de largo batch_size
    v = [encode(x[1]) for x in b]
    l = max(map(len, v))
    return ( # tupla de dos tensores: etiquetas, características rellenadas
        torch.LongTensor([t[0] for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

def train_epoch(net,dataloader,lr=0.01,optimizer=None,loss_fn=torch.nn.CrossEntropyLoss(),epoch_size=None,report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,features in dataloader:
        optimizer.zero_grad()
        features, labels = features.to(device), labels.to(device)
        out = net(features)
        loss = loss_fn(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn=torch.nn.CrossEntropyLoss(),epoch_size=None,report_freq=200,use_pack_sequence=False):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,lengths in dataloader:
        optimizer.zero_grad()
        labels,text = labels.to(device), text.to(device)
        # pack_padded_sequence exige que el tensor de largos esté en la CPU
        lengths = lengths.cpu() if use_pack_sequence else lengths.to(device)
        out = net(text, lengths)
        loss = loss_fn(out, labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count

## Clasificador RNN simple

En el caso de una RNN simple, cada unidad recurrente es una red lineal sencilla que recibe la concatenación del vector de entrada y el vector de estado, y produce un nuevo vector de estado. PyTorch representa esta unidad con la clase `RNNCell`, y una red de estas celdas con la capa `RNN`.

Para definir un clasificador RNN, primero aplicaremos una capa de embedding para reducir la dimensionalidad del vocabulario de entrada, y luego pondremos una capa RNN sobre ella: 

In [5]:
class RNNClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.rnn = torch.nn.RNN(embed_dim,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, num_class)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.embedding(x)
        x,h = self.rnn(x)
        return self.fc(x.mean(dim=1))

> **Nota:** Aquí usamos por simplicidad una capa de embedding sin entrenar, pero para obtener resultados aún mejores podemos usar una capa de embedding preentrenada con embeddings Word2Vec o GloVe, como se describió en la unidad anterior. Para comprenderlo mejor, puedes adaptar este código para que funcione con embeddings preentrenados.

En nuestro caso usaremos un data loader con relleno (padding), de modo que cada batch contendrá varias secuencias rellenadas de la misma longitud. La capa RNN recibirá la secuencia de tensores de embedding y producirá dos salidas:
* $x$ es la secuencia de salidas de la celda RNN en cada paso
* $h$ es el estado oculto final para el último elemento de la secuencia

Luego aplicamos un clasificador lineal totalmente conectado para obtener el número de la clase.

> **Nota:** Las RNN son bastante difíciles de entrenar, porque una vez que las celdas RNN se desenrollan a lo largo de la secuencia, el número de capas involucradas en la retropropagación es bastante grande. Por eso debemos elegir una tasa de aprendizaje pequeña y entrenar la red con un dataset más grande para obtener buenos resultados. Esto puede tomar bastante tiempo, por lo que es preferible usar GPU.

In [6]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, collate_fn=padify, shuffle=True)
net = RNNClassifier(vocab_size,64,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=0.001)

12800: acc=0.86171875


25600: acc=0.916640625


38400: acc=0.9395833333333333


51200: acc=0.95169921875


64000: acc=0.959359375


...

256000: acc=0.9868046875


268800: acc=0.9872284226190476


281600: acc=0.9877024147727272


294400: acc=0.9881317934782609


(0.0005023447839907401, 0.9883557044748932)

## Long Short Term Memory (LSTM)

Uno de los principales problemas de las RNN clásicas es el llamado problema de **desvanecimiento del gradiente** (*vanishing gradients*). Como las RNN se entrenan de extremo a extremo en una sola pasada de retropropagación, les cuesta propagar el error hasta las primeras capas de la red, y por lo tanto la red no puede aprender relaciones entre tokens distantes. Una forma de evitar este problema es introducir un **manejo explícito del estado** mediante las llamadas **compuertas** (*gates*). Las dos arquitecturas más conocidas de este tipo son **Long Short Term Memory** (LSTM) y **Gated Recurrent Unit** (GRU).

![Imagen que muestra un ejemplo de celda long short term memory](./images/long-short-term-memory-cell.svg)

Una red LSTM se organiza de forma similar a una RNN, pero hay dos estados que se pasan de capa en capa: el estado propiamente tal $c$ y el vector oculto $h$. En cada unidad, el vector oculto $h_i$ se concatena con la entrada $x_i$, y juntos controlan lo que le ocurre al estado $c$ a través de las **compuertas**. Cada compuerta es una red neuronal con activación sigmoide (salida en el rango $[0,1]$), que puede pensarse como una máscara bit a bit cuando se multiplica por el vector de estado. Existen las siguientes compuertas (de izquierda a derecha en la figura anterior):
* la **compuerta de olvido** (*forget gate*) toma el vector oculto y determina qué componentes del vector $c$ debemos olvidar y cuáles dejar pasar.
* la **compuerta de entrada** (*input gate*) toma información de la entrada y del vector oculto, y la inserta en el estado.
* la **compuerta de salida** (*output gate*) transforma el estado mediante una capa lineal con activación $\tanh$, y luego selecciona algunas de sus componentes usando el vector oculto $h_i$ para producir el nuevo estado $c_{i+1}$.

Las componentes del estado $c$ pueden pensarse como banderas que se pueden activar y desactivar. Por ejemplo, cuando encontramos el nombre *Alice* en la secuencia, podemos suponer que se refiere a un personaje femenino y activar en el estado la bandera que indica que hay un sustantivo femenino en la oración. Si más adelante encontramos la frase *and Tom* (y Tom), activaremos la bandera que indica que tenemos un sustantivo plural. Así, manipulando el estado, en principio podemos llevar registro de las propiedades gramaticales de las partes de la oración.

> **Nota**: Un excelente recurso para entender el funcionamiento interno de una LSTM es el artículo [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) de Christopher Olah.

Aunque la estructura interna de una celda LSTM puede parecer compleja, PyTorch oculta esta implementación dentro de la clase `LSTMCell` y ofrece el objeto `LSTM` para representar una capa LSTM completa. Por lo tanto, la implementación de un clasificador LSTM será bastante similar a la de la RNN simple que vimos antes:

In [7]:
class LSTMClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data = torch.randn_like(self.embedding.weight.data)-0.5
        self.rnn = torch.nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, num_class)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.embedding(x)
        x,(h,c) = self.rnn(x)
        return self.fc(h[-1])

Ahora entrenemos nuestra red. Ten en cuenta que entrenar una LSTM también es bastante lento, y puede que no veas un gran aumento en la exactitud (accuracy) al comienzo del entrenamiento. Además, quizás debas experimentar con el parámetro de tasa de aprendizaje `lr` para encontrar un valor que dé una velocidad de entrenamiento razonable sin desperdiciar memoria.

In [8]:
net = LSTMClassifier(vocab_size,64,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=0.001)

12800: acc=0.770625


25600: acc=0.873046875


38400: acc=0.9117447916666667


51200: acc=0.931484375


64000: acc=0.9435


...

256000: acc=0.98327734375


268800: acc=0.9840141369047619


281600: acc=0.9846803977272728


294400: acc=0.9852547554347826


(0.000480635573827416, 0.9855401982488682)

## Secuencias empaquetadas (packed sequences)

En nuestro ejemplo tuvimos que rellenar todas las secuencias del minibatch con vectores de ceros. Aunque esto implica cierto desperdicio de memoria, en las RNN lo más crítico es que se crean celdas RNN adicionales para los elementos de relleno, que participan en el entrenamiento pero no aportan información relevante de la entrada. Sería mucho mejor entrenar la RNN solo hasta el largo real de cada secuencia.

Para ello, PyTorch introduce un formato especial de almacenamiento de secuencias rellenadas. Supongamos que tenemos un minibatch de entrada rellenado como este:
```
[[1,2,3,4,5],
 [6,7,8,0,0],
 [9,0,0,0,0]]
```
Aquí 0 representa los valores de relleno, y el vector con los largos reales de las secuencias de entrada es `[5,3,1]`.

Para entrenar eficientemente una RNN con secuencias rellenadas, queremos comenzar a entrenar el primer grupo de celdas RNN con un minibatch grande (`[1,6,9]`), luego terminar el procesamiento de la tercera secuencia y continuar entrenando con minibatches más cortos (`[2,7]`, `[3,8]`), y así sucesivamente. Así, una secuencia empaquetada se representa como un solo vector —en nuestro caso `[1,6,9,2,7,3,8,4,5]`— y un vector de largos (`[5,3,1]`), a partir de los cuales podemos reconstruir fácilmente el minibatch rellenado original.

Para producir una secuencia empaquetada podemos usar la función `torch.nn.utils.rnn.pack_padded_sequence`. Todas las capas recurrentes, incluidas RNN, LSTM y GRU, aceptan secuencias empaquetadas como entrada y producen una salida empaquetada, que puede decodificarse con `torch.nn.utils.rnn.pad_packed_sequence`.

Para poder producir una secuencia empaquetada debemos pasar el vector de largos a la red, por lo que necesitamos una función distinta para preparar los minibatches:

In [9]:
def pad_length(b):
    # construir la secuencia vectorizada
    v = [encode(x[1]) for x in b]
    # calcular el largo máximo de una secuencia en este minibatch y la propia secuencia de largos
    len_seq = list(map(len,v))
    l = max(len_seq)
    return ( # tupla de tres tensores: etiquetas, características rellenadas, secuencia de largos
        torch.LongTensor([t[0] for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v]),
        torch.tensor(len_seq)
    )

train_loader_len = torch.utils.data.DataLoader(train_dataset, batch_size=64, collate_fn=pad_length, shuffle=True)

La red propiamente tal será muy similar a `LSTMClassifier` de arriba, pero la pasada `forward` recibirá tanto el minibatch rellenado como el vector de largos de las secuencias. Después de calcular el embedding, calculamos la secuencia empaquetada, la pasamos a la capa LSTM y luego desempaquetamos el resultado.

> **Nota**: En realidad no usamos el resultado desempaquetado `x`, porque en los cálculos siguientes usamos la salida de las capas ocultas. Por lo tanto, podríamos eliminar por completo el desempaquetado de este código. Lo dejamos aquí para que puedas modificar el código fácilmente en caso de que necesites usar la salida de la red en cálculos posteriores.

In [10]:
class LSTMPackClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_class):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight.data = torch.randn_like(self.embedding.weight.data)-0.5
        self.rnn = torch.nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, num_class)

    def forward(self, x, lengths):
        batch_size = x.size(0)
        x = self.embedding(x)
        pad_x = torch.nn.utils.rnn.pack_padded_sequence(x,lengths,batch_first=True,enforce_sorted=False)
        pad_x,(h,c) = self.rnn(pad_x)
        x, _ = torch.nn.utils.rnn.pad_packed_sequence(pad_x,batch_first=True)
        return self.fc(h[-1])

Ahora hagamos el entrenamiento:

In [11]:
net = LSTMPackClassifier(vocab_size,64,32,len(classes)).to(device)
train_epoch_emb(net,train_loader_len, lr=0.001,use_pack_sequence=True)


12800: acc=0.881640625


25600: acc=0.9337890625


38400: acc=0.95296875


51200: acc=0.9631640625


64000: acc=0.969484375


...

256000: acc=0.9904453125


268800: acc=0.9908407738095238


281600: acc=0.9912109375


294400: acc=0.991484375


(0.0003228979674298419, 0.9916465559079094)

> **Nota:** Quizás notaste el parámetro `use_pack_sequence` que pasamos a la función de entrenamiento. Actualmente, la función `pack_padded_sequence` requiere que el tensor con los largos de las secuencias esté en la CPU, por lo que la función de entrenamiento debe evitar mover esos datos a la GPU durante el entrenamiento. Puedes revisar la implementación de la función `train_epoch_emb` en las funciones auxiliares definidas al comienzo de este notebook.

## RNN bidireccionales y multicapa

En nuestros ejemplos, todas las redes recurrentes operaron en una sola dirección, desde el comienzo de la secuencia hasta el final. Parece natural, porque se asemeja a la forma en que leemos y escuchamos el habla. Sin embargo, como en muchos casos prácticos tenemos acceso aleatorio a la secuencia de entrada, puede tener sentido ejecutar el cómputo recurrente en ambas direcciones. Estas redes se llaman RNN **bidireccionales**, y pueden crearse pasando el parámetro `bidirectional=True` al constructor de RNN/LSTM/GRU.

Al trabajar con una red bidireccional necesitamos dos vectores de estado oculto, uno por cada dirección. PyTorch codifica esos vectores como un solo vector del doble de tamaño, lo que es bastante conveniente, porque normalmente el estado oculto resultante se entrega a una capa lineal totalmente conectada, y solo hay que considerar este aumento de tamaño al crear dicha capa.

Una red recurrente, unidireccional o bidireccional, captura ciertos patrones dentro de una secuencia, y puede almacenarlos en el vector de estado o pasarlos a la salida. Al igual que con las redes convolucionales, podemos construir otra capa recurrente sobre la primera para capturar patrones de más alto nivel, construidos a partir de los patrones de bajo nivel extraídos por la primera capa. Esto nos lleva a la noción de **RNN multicapa**, que consiste en dos o más redes recurrentes, donde la salida de la capa anterior se entrega como entrada a la capa siguiente.

![Imagen que muestra una RNN long-short-term-memory multicapa](images/multi-layer-lstm.jpg)

*Figura tomada de [este excelente post](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) de Fernando López*

PyTorch facilita la construcción de estas redes, ya que basta con pasar el parámetro `num_layers` al constructor de RNN/LSTM/GRU para construir automáticamente varias capas de recurrencia. Esto también significa que el tamaño del vector oculto/de estado aumentará proporcionalmente, y habrá que tenerlo en cuenta al manejar la salida de las capas recurrentes.

## RNN para otras tareas

En esta unidad vimos que las RNN pueden usarse para clasificar secuencias, pero en realidad pueden abordar muchas otras tareas, como la generación de texto, la traducción automática y más. Veremos esas tareas en la próxima unidad.